# Goal-scoring opportunity phase: defensive compactness and space control

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

During the goal-scoring opportunity phase, the compactness of the defensive formation is evaluated by measuring inter-player distances, and space control is analysed using Voronoi diagrams.

This notebook runs the shared pipeline (`pitchvision`: detection, tracking, pitch calibration, team classification - see `00_pipeline_demo.ipynb` for a step-by-step walkthrough of each of those with sanity checks) condensed into one setup section, then applies two spatial analyses to a clip from the `Goals` folder:

1. **Defensive compactness**: mean inter-player distance and "stretch" (average distance from the team centroid) for the defending team, over the course of the phase.
2. **Space control**: a Voronoi tessellation of the pitch by every player's position, crediting each player (and therefore each team) with the pitch area closer to them than to anyone else.

## 1. Setup

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/football-spatial-analysis-thesis-lx13x7"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log -1 --oneline

# Put the package on sys.path directly, rather than relying on `pip install -e .`
# to register it - editable installs use a .pth/import-finder file that Python's
# site module only processes at interpreter startup, so one run mid-session (in
# an already-running Colab kernel) doesn't reliably become importable.
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Drive and pick a clip

In [ ]:
from pitchvision import DriveConfig, list_videos, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive/Magisterka")
goal_videos = list_videos(drive_cfg.goals_path)
print(f"Found {len(goal_videos)} videos in Goals/")

sample_video = goal_videos[0]  # change the index to analyse a different clip
sample_video

## 3. Run the shared pipeline

Condensed: pitch calibration (automatic, from detected pitch keypoints) and team classification (jersey colour, using the specialized player/goalkeeper/referee detector). If either step needs troubleshooting for this particular clip, do it in `00_pipeline_demo.ipynb` first - it has the sanity-check visualisations and the manual-calibration fallback.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from pitchvision import PitchKeypointDetector, VideoFrames, download_pitch_keypoint_weights

frames = VideoFrames(sample_video)
first_frame = frames.read_frame(0)

pitch_weights_path = download_pitch_keypoint_weights(
    "/content/drive/MyDrive/pitchvision_models/football-pitch-detection.pt"
)
keypoint_detector = PitchKeypointDetector(weights=pitch_weights_path, confidence=0.5)
calibrator = keypoint_detector.calibrate(first_frame)
print("Pitch calibrated.")

In [ ]:
from pitchvision import (
    PlayerBallDetector,
    SPORTS_DETECTION_CLASSES,
    TeamClassifier,
    collect_jersey_colors,
    download_player_detection_weights,
)

player_weights_path = download_player_detection_weights(
    "/content/drive/MyDrive/pitchvision_models/football-player-detection.pt"
)
player_detector = PlayerBallDetector(
    weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES
)

colors = collect_jersey_colors(sample_video, player_detector, class_names=("player",), stride=15)
team_classifier = TeamClassifier(n_clusters=2)
team_classifier.fit(colors)

swatches = team_classifier.cluster_swatches
fig, axes = plt.subplots(1, len(swatches), figsize=(4 * len(swatches), 2))
for team_id, (ax, rgb) in enumerate(zip(axes, swatches)):
    ax.imshow([[rgb]])
    ax.set_title(f"team_id = {team_id}")
    ax.axis("off")
plt.show()

**Identify the defending team.** Look at the swatch colours above against the clip: the goal-scoring opportunity phase is defined from the perspective of the team *conceding* the chance. Set `DEFENDING_TEAM_ID` below to that team's `team_id`.

In [ ]:
DEFENDING_TEAM_ID = 0  # EDIT THIS based on the swatches above

In [ ]:
from pitchvision import PlayerTracker, TrackingPipeline

tracker = PlayerTracker(weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES)
pipeline = TrackingPipeline(
    tracker=tracker,
    calibrator=calibrator,
    team_classifier=team_classifier,
    team_eligible_class_names=("player",),
)

tracks_df = pipeline.run(sample_video)  # a Goals clip is short enough to run in full
tracks_df["class_name"].value_counts()

## 4. Defensive compactness (inter-player distances)

`pitchvision.compactness.compute_team_compactness` gives one row per frame with the defending team's mean pairwise inter-player distance and "stretch index" (mean distance from the team centroid) - a lower value means a tighter, more compact defensive block.

In [ ]:
from pitchvision import compute_team_compactness

compactness_df = compute_team_compactness(tracks_df, team_id=DEFENDING_TEAM_ID)
compactness_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(compactness_df["frame"], compactness_df["mean_pairwise_distance_m"], label="Mean pairwise distance (m)")
ax.plot(compactness_df["frame"], compactness_df["stretch_index_m"], label="Stretch index (m)")
ax.set_xlabel("Frame")
ax.set_ylabel("Metres")
ax.set_title("Defensive compactness over the phase")
ax.legend()
plt.show()

print(compactness_df[["mean_pairwise_distance_m", "stretch_index_m", "length_m", "width_m"]].describe())

## 5. Space control (Voronoi diagrams)

`pitchvision.voronoi.pitch_voronoi_cells` tessellates the pitch by every tracked player's position (both teams): the region closer to a given player than to anyone else is credited to them. This is the standard simplified "space control" model in tactical analysis - it ignores player speed/orientation/reaction time, unlike more advanced pitch-control models, but is a well-established first-order approximation.

In [ ]:
from pitchvision import draw_pitch, pitch_voronoi_cells, plot_voronoi

TEAM_COLORS = {0: "yellow", 1: "red"}

# A representative frame: the one with the most on-pitch players tracked.
on_pitch = tracks_df[tracks_df["class_name"].isin(["player", "goalkeeper"]) & tracks_df["team_id"].notna()]
sample_frame = on_pitch.groupby("frame").size().idxmax()
frame_rows = on_pitch[on_pitch["frame"] == sample_frame]

positions = frame_rows[["pitch_x", "pitch_y"]].to_numpy()
team_ids = frame_rows["team_id"].to_numpy()
polygons = pitch_voronoi_cells(positions)

ax = draw_pitch()
plot_voronoi(ax, polygons, team_ids, team_colors=TEAM_COLORS)
for (x, y), team_id in zip(positions, team_ids):
    ax.scatter(x, y, color=TEAM_COLORS.get(team_id, "gray"), edgecolors="black", s=60, zorder=3)
plt.title(f"Space control at frame {sample_frame}")
plt.show()

In [ ]:
from pitchvision import PITCH_LENGTH_M, PITCH_WIDTH_M, compute_space_control

space_df = compute_space_control(tracks_df)
pitch_area_m2 = PITCH_LENGTH_M * PITCH_WIDTH_M

fig, ax = plt.subplots(figsize=(10, 4))
for team_id in (0, 1):
    col = f"team_{team_id}_area_m2"
    if col in space_df.columns:
        ax.plot(space_df["frame"], 100 * space_df[col] / pitch_area_m2,
                 label=f"Team {team_id}", color=TEAM_COLORS.get(team_id, "gray"))
ax.set_xlabel("Frame")
ax.set_ylabel("% of pitch controlled")
ax.set_title("Space control over the phase")
ax.legend()
plt.show()

defending_col = f"team_{DEFENDING_TEAM_ID}_area_m2"
if defending_col in space_df.columns:
    mean_pct = 100 * space_df[defending_col].mean() / pitch_area_m2
    print(f"Defending team (team_id={DEFENDING_TEAM_ID}) controlled {mean_pct:.1f}% of the pitch on average.")

## 6. Save results

In [ ]:
output_dir = "/content/drive/MyDrive/pitchvision_outputs"
os.makedirs(output_dir, exist_ok=True)
clip_name = os.path.splitext(os.path.basename(sample_video))[0]

compactness_path = os.path.join(output_dir, f"{clip_name}_compactness.csv")
space_control_path = os.path.join(output_dir, f"{clip_name}_space_control.csv")
compactness_df.to_csv(compactness_path, index=False)
space_df.to_csv(space_control_path, index=False)
compactness_path, space_control_path

## Notes and limitations

- The Voronoi space-control model treats every player as controlling the region strictly closer to them by straight-line distance. It doesn't account for player speed, current motion, or reaction time the way more advanced "pitch control" models (e.g. Spearman et al.) do - two players equidistant from a point are credited equally even if one is already sprinting toward it and the other is facing the wrong way. Treat the area/percentage figures as a first-order approximation, not a physically-grounded probability of reaching the ball first.
- Goalkeepers are included in space control (they occupy real pitch space) but excluded from the defensive-compactness metric by default (`compute_team_compactness`'s `class_names` doesn't include `"goalkeeper"`), since including a goalkeeper anchored near their own goal line would distort an outfield defensive-line compactness measure.
- To compare compactness/space-control figures across multiple goal-scoring opportunity clips for the thesis's quantitative analysis, re-run this notebook per clip (change `sample_video` in section 2) and aggregate the saved CSVs.